# Module 2 Validation — classifai_validation_data

This notebook validates the inference pipeline on the classifai validation dataset.
Queries are built by concatenating `field_job_description` and `field_occupation_description`.
Targets are derived from the deepest non-empty code per taxonomy level.


In [ ]:
from pathlib import Path
import pandas as pd

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
from kedro.runner import SequentialRunner

from taxomind.pipeline_registry import register_pipelines

project_path = Path.cwd()
if not (project_path / "conf").exists():
    project_path = project_path.parent
bootstrap_project(project_path)


In [ ]:
import contextlib
import io
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for name in ["kedro", "taxomind", "sentence_transformers", "transformers", "urllib3"]:
    logging.getLogger(name).setLevel(logging.ERROR)


In [ ]:
# Load validation data
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    df_raw = context.catalog.load("classifai_validation_data")

df = df_raw.copy()
df = df.reset_index(drop=True)
df["query_id"] = df.index


In [ ]:
# Normalize code columns (digits only; otherwise empty)
CODE_COLS = [
    "ISCO_1", "ISCO_2", "ISCO_3", "ISCO_4",
    "ISIC_1", "ISIC_2", "ISIC_3", "ISIC_4",
]

def normalize_code_series(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    return s.where(s.str.fullmatch(r"\d+"), "")

for col in CODE_COLS:
    if col in df.columns:
        df[col] = normalize_code_series(df[col])


In [ ]:
# Build query text from job + occupation description
job = df.get("field_job_description", pd.Series([""] * len(df))).fillna("").astype(str)
occ = df.get("field_occupation_description", pd.Series([""] * len(df))).fillna("").astype(str)
df["query_text"] = (job + " " + occ).str.replace(r"\s+", " ", regex=True).str.strip()


In [ ]:
# Derive deepest target per taxonomy
def pick_target(row, prefix: str):
    for level in (4, 3, 2, 1):
        val = str(row.get(f"{prefix}_{level}", "")).strip()
        if val:
            return val, level
    return None, None

df[["target_isco_code", "target_isco_level"]] = df.apply(
    lambda r: pd.Series(pick_target(r, "ISCO")), axis=1
)
df[["target_isic_code", "target_isic_level"]] = df.apply(
    lambda r: pd.Series(pick_target(r, "ISIC")), axis=1
)


In [ ]:
# Run inference for a taxonomy key
def run_inference(taxonomy_key: str, queries):
    buf_out = io.StringIO()
    buf_err = io.StringIO()
    with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
        with KedroSession.create(
            project_path=project_path,
            runtime_params={
                "taxonomy_key": taxonomy_key,
                "inference_query_input": queries,
            },
        ) as session:
            run_result = session.run(pipeline_name="inference")
            preds = run_result["inference_predictions_df"].load()

    preds["taxonomy_key"] = taxonomy_key
    print(f"{taxonomy_key} inference done: {len(preds)} rows")
    return preds

queries = df["query_text"].tolist()


In [ ]:
preds_isco = run_inference("ISCO", queries)

In [ ]:
preds_isic = run_inference("ISIC", queries)

In [ ]:
# Build parent/level maps for ancestor checks
def load_taxonomy_maps(taxonomy_key: str):
    with KedroSession.create(project_path=project_path) as session:
        context = session.load_context()
        partitions = context.catalog.load("taxonomy_index")
    tax_df = partitions[taxonomy_key]()
    parent_map = {}
    level_map = {}
    for _, row in tax_df.iterrows():
        parent = row["parentCode"]
        if pd.isna(parent) or parent == "":
            parent = "__root__"
        code = str(row["code"]).strip()
        parent_map[code] = str(parent).strip()
        level_map[code] = int(row["level"])
    return parent_map, level_map

def is_ancestor(ancestor: str, code: str, parent_map: dict) -> bool:
    if not ancestor or not code or ancestor == "__root__":
        return False
    current = code
    while current and current != "__root__":
        if current == ancestor:
            return True
        current = parent_map.get(current)
    return False

isco_parent_map, isco_level_map = load_taxonomy_maps("ISCO")
isic_parent_map, isic_level_map = load_taxonomy_maps("ISIC")


In [ ]:
# Fill ISIC_1 from deeper ISIC codes using taxonomy parent map
def find_level1(code: str, parent_map: dict, level_map: dict) -> str:
    current = str(code).strip()
    while current and current != '__root__':
        if level_map.get(current) == 1:
            return current
        current = parent_map.get(current)
    return ''

def derive_isic1(row) -> str:
    for level in (4, 3, 2, 1):
        val = str(row.get(f'ISIC_{level}', '')).strip()
        if val or val == '':
            return find_level1(val, isic_parent_map, isic_level_map)
    return ''

df['ISIC_1'] = df.apply(
    lambda r: r['ISIC_1'] if str(r.get('ISIC_1', '')).strip() else derive_isic1(r),
    axis=1,
)


In [ ]:
# Expand predicted codes to level-wise columns and compare
def expand_predicted_levels(preds, taxonomy_key, parent_map, level_map, max_level=4):
    code_levels = {}
    for code in preds["predicted_code"].dropna().unique():
        code = str(code).strip()
        levels = {}
        current = code
        while current and current != "__root__":
            level = level_map.get(current)
            if level is not None:
                levels[int(level)] = current
            current = parent_map.get(current)
        code_levels[code] = levels

    preds = preds.copy()
    for level in range(1, max_level + 1):
        col = f"pred_{taxonomy_key}_{level}"
        preds[col] = preds["predicted_code"].map(
            lambda c: code_levels.get(str(c).strip(), {}).get(level, "")
        )
    return preds

preds_isco = expand_predicted_levels(
    preds_isco, "ISCO", isco_parent_map, isco_level_map
)
preds_isic = expand_predicted_levels(
    preds_isic, "ISIC", isic_parent_map, isic_level_map
)

def level_match_summary(preds, targets_df, taxonomy_key):
    rows = []
    for level in range(1, 5):
        target_col = f"{taxonomy_key}_{level}"
        pred_col = f"pred_{taxonomy_key}_{level}"
        merged = preds.merge(
            targets_df[["query_id", target_col]],
            on="query_id",
            how="left",
        )
        valid = merged[target_col].fillna("").str.strip() != ""
        match_rate = (merged.loc[valid, pred_col] == merged.loc[valid, target_col]).mean() if valid.any() else 0.0
        rows.append({
            "taxonomy": taxonomy_key,
            "level": level,
            "rows": int(valid.sum()),
            "match_rate": match_rate,
        })
    return pd.DataFrame(rows)

isco_level_match = level_match_summary(preds_isco, df, "ISCO")
isic_level_match = level_match_summary(preds_isic, df, "ISIC")

isco_level_match


In [ ]:
isic_level_match

In [ ]:
# Evaluation helpers
def evaluate(preds, targets_df, taxonomy_key, parent_map):
    if taxonomy_key == "ISCO":
        target_code_col = "target_isco_code"
        target_level_col = "target_isco_level"
    else:
        target_code_col = "target_isic_code"
        target_level_col = "target_isic_level"

    merged = preds.merge(
        targets_df[["query_id", target_code_col, target_level_col]],
        on="query_id",
        how="left",
    )
    valid = merged[merged[target_code_col].fillna("").str.strip() != ""].copy()

    valid["exact_match"] = valid["predicted_code"] == valid[target_code_col]
    valid["level_match"] = valid["predicted_level"] == valid[target_level_col]

    valid["under_spec"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r[target_code_col], parent_map)
        and r["predicted_code"] != r[target_code_col],
        axis=1,
    )
    valid["over_spec"] = valid.apply(
        lambda r: is_ancestor(r[target_code_col], r["predicted_code"], parent_map)
        and r["predicted_code"] != r[target_code_col],
        axis=1,
    )
    valid["ancestor_match"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r[target_code_col], parent_map)
        or r["predicted_code"] == r[target_code_col],
        axis=1,
    )

    summary = {
        "taxonomy": taxonomy_key,
        "rows_evaluated": len(valid),
        "exact_match_rate": valid["exact_match"].mean() if len(valid) else 0.0,
        "level_match_rate": valid["level_match"].mean() if len(valid) else 0.0,
        "ancestor_match_rate": valid["ancestor_match"].mean() if len(valid) else 0.0,
        "over_spec_rate": valid["over_spec"].mean() if len(valid) else 0.0,
        "under_spec_rate": valid["under_spec"].mean() if len(valid) else 0.0,
        "ambiguous_rate": valid["ambiguous"].mean() if len(valid) else 0.0,
    }

    return valid, summary


In [ ]:
isco_eval, isco_summary = evaluate(preds_isco, df, "ISCO", isco_parent_map)
isic_eval, isic_summary = evaluate(preds_isic, df, "ISIC", isic_parent_map)

summary_df = pd.DataFrame([isco_summary, isic_summary])
summary_df


In [ ]:
# Level-wise evaluation
def level_summary(eval_df, level_col):
    grouped = eval_df.groupby(level_col)
    summary = grouped.agg(
        rows=("query_id", "size"),
        exact_match_rate=("exact_match", "mean"),
        ancestor_match_rate=("ancestor_match", "mean"),
        over_spec_rate=("over_spec", "mean"),
        under_spec_rate=("under_spec", "mean"),
        ambiguous_rate=("ambiguous", "mean"),
    ).reset_index()
    return summary.rename(columns={level_col: "target_level"})

isco_level_summary = level_summary(isco_eval, "target_isco_level")
isic_level_summary = level_summary(isic_eval, "target_isic_level")

isco_level_summary


In [ ]:
isic_level_summary

In [ ]:
isco_eval

In [ ]:
isic_eval

In [ ]:
isic_eval

In [ ]:
df

In [ ]:
# Validation status distribution
isco_eval["validation_status"].value_counts(dropna=False)


In [ ]:
isic_eval["validation_status"].value_counts(dropna=False)


In [ ]:
# find broken parent links
tax_df = context.catalog.load("taxonomy_index")["ISCO"]()
codes = set(tax_df["code"].astype(str).str.strip())

bad = tax_df[
    tax_df["parentCode"].notna()
    & (tax_df["parentCode"].astype(str).str.strip() != "__root__")
    & (~tax_df["parentCode"].astype(str).str.strip().isin(codes))
]
bad[["code", "parentCode"]].head()


In [ ]:
from collections import defaultdict
tax_df = context.catalog.load("taxonomy_index")["ISIC"]()
taxonomy_graph = defaultdict(list)
for _, row in tax_df.iterrows():
    parent = row["parentCode"]
    if pd.isna(parent) or parent == "":
        parent = "__root__"
    taxonomy_graph[parent].append(row["code"])
taxonomy_graph = dict(taxonomy_graph)

root_children = taxonomy_graph.get("__root__", [])
root_children#[c for c in root_children if c in candidates_dict["V_codes"]]
